In [1]:
import pymupdf
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
sources = pd.read_csv("../data/sources.csv")

path = RAW_DIR / sources.loc[sources["ref_no"] == "BOFIA 2020", "filename"].iloc[0]

doc = pymupdf.open(path)
print("Pages:", len(doc))
print("Metadata title:", doc.metadata.get("title"))
print()
print(doc[6].get_text()[:1200])
doc.close()

Pages: 88
Metadata title: 




In [2]:
def page_text_stats(path, sample_pages=6):
    """Measure how much extractable text a PDF has."""
    doc = pymupdf.open(path)
    n = len(doc)
    idx = list(range(min(n, sample_pages))) if n <= sample_pages else \
          [int(i * (n - 1) / (sample_pages - 1)) for i in range(sample_pages)]

    chars = [len(doc[i].get_text().strip()) for i in idx]
    doc.close()
    return {"pages": n, "avg_chars_per_page": round(sum(chars) / len(chars), 1), "min_chars": min(chars)}


rows = []
for _, r in sources.iterrows():
    path = RAW_DIR / r["filename"]
    try:
        stats = page_text_stats(path)
        stats["kind"] = "SCANNED" if stats["avg_chars_per_page"] < 100 else "text"
        stats["error"] = ""
    except Exception as exc:
        stats = {"pages": 0, "avg_chars_per_page": 0, "min_chars": 0,
                 "kind": "UNREADABLE", "error": f"{type(exc).__name__}: {exc}"[:80]}
    stats.update({"ref_no": r["ref_no"], "regulator": r["regulator"], "title": r["title"][:50]})
    rows.append(stats)

audit = pd.DataFrame(rows)[["regulator", "ref_no", "title", "pages", "avg_chars_per_page", "kind", "error"]]

print(audit["kind"].value_counts().to_string())
print("\nTotal pages:", int(audit["pages"].sum()))
print()
print(audit.sort_values("avg_chars_per_page").to_string(index=False))

kind
text       18
SCANNED     6

Total pages: 1231

regulator                  ref_no                                              title  pages  avg_chars_per_page    kind error
      CBN FPR/DIR/PUB/CIR/002/002 Additional Know Your Customer Requirement in Respe      1                 0.0 SCANNED      
      CBN FPR/DIR/PUB/CIR/001/014 RE: GUIDELINES ON MANAGEMENT OF DORMANT ACCOUNTS,       3                 0.0 SCANNED      
      CBN BSD/DIR/PUB/LAB/017/003 Re: Impact of Recent Policy Reforms-Prudential Gui      1                 0.0 SCANNED      
      CBN              BOFIA 2020    Banks and Other Financial Institutions Act 2020     88                 0.0 SCANNED      
      CBN  FPR/DIR/GEN/CIR/01/004 Circular to All Banks and Other Financial Institut     20                 0.0 SCANNED      
      CBN  GVD/ESO/GEN/PMF/02/004 Anti-Money Laundering/Combating the Financing of T     37                 0.0 SCANNED      
      CBN FPR/DIR/PUB/CIR/002/009 Review of Minimum Capital Requi

In [3]:
import io
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 5.5.3.20260724


In [4]:
def ocr_page(page, dpi=300):
    """Render a PDF page as an image and read the text off it."""
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(img)


bofia = RAW_DIR / sources.loc[sources["ref_no"] == "BOFIA 2020", "filename"].iloc[0]

doc = pymupdf.open(bofia)
print("OCR of page 7 of", len(doc), "\n")
text = ocr_page(doc[6])
doc.close()

print(text[:1800])

OCR of page 7 of 88 

Banks and Other Financial Institutions Act, 2020 2020 No. 5 A 655

120. Account and audit.

121. Application of Tribunal’s Fund.

122. Powers of the Tribunal and President of the Tribunal.
123. Panel of experts.

124. Request for an expert.

125. Right to legal representation.

126. Powers of the Tribunal and President of the Tribunal.
127. Appeal to the Court of Appeal.

128. Further appeals.

129, Protection of members of the Tribunal.

130. Repeal.

131. Interpretation.

132. Citation.

eh



In [5]:
import json
from tqdm.auto import tqdm

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)


def extract_document(path, use_ocr, dpi=300):
    """Return a list of pages, using OCR for pages with no extractable text."""
    doc = pymupdf.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        method = "text"
        if use_ocr and len(text) < 100:
            text = ocr_page(page, dpi=dpi).strip()
            method = "ocr"
        pages.append({"page": i + 1, "method": method, "text": text})
    doc.close()
    return pages


for _, r in tqdm(list(sources.iterrows()), desc="extracting"):
    out_path = PROCESSED / (Path(r["filename"]).stem + ".json")
    if out_path.exists():
        continue

    pdf_path = RAW_DIR / r["filename"]
    use_ocr = page_text_stats(pdf_path)["avg_chars_per_page"] < 100

    pages = extract_document(pdf_path, use_ocr)

    out_path.write_text(json.dumps({
        "filename": r["filename"],
        "regulator": r["regulator"],
        "ref_no": r["ref_no"] if pd.notna(r["ref_no"]) else "",
        "title": r["title"],
        "url": r["url"],
        "document_date": r["document_date"] if pd.notna(r["document_date"]) else "",
        "ocr_used": use_ocr,
        "pages": pages,
    }, ensure_ascii=False), encoding="utf-8")

print("\nExtracted files:", len(list(PROCESSED.glob("*.json"))))

extracting:   0%|          | 0/24 [00:00<?, ?it/s]


Extracted files: 24


In [6]:
data = json.loads((PROCESSED / (Path(bofia.name).stem + ".json")).read_text(encoding="utf-8"))

print("OCR used:", data["ocr_used"], "| pages:", len(data["pages"]))
print("Methods:", pd.Series([p["method"] for p in data["pages"]]).value_counts().to_dict())
print("\n--- page 30 ---\n")
print(data["pages"][29]["text"][:1800])

OCR used: True | pages: 88
Methods: {'ocr': 88}

--- page 30 ---

A 678

Contents
and form of
accounts,

2020 No. § Banks and Other Financial Institutions Act, 2020

(2) Every bank or other financial institution shall thereafter, but not later
than seven days after approval for publication by the Bank—

(a) cause to be published in not less than two national daily newspapers
printed and circulating in Nigeria ;

(5) exhibit in a conspicuous position in each of its offices, branches and
website ; and

(c) forward to the Bank, copies of the bank's published Statement of
Financial Position and Statement of Profit and Loss and Other
Comprehensive Income duly signed with the full names of the directors
of the bank who signed the financial Statements, and in the case of a non-
interest bank, a copy of the report of tts advisory committee of experts,

(3) Every published account of'a bank under subsection (2), shall disclose
in detail penalties paid for the contravention of the provistons of 

In [7]:
def ocr_page_cfg(page, dpi=300, config=""):
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(img, config=config)


doc = pymupdf.open(bofia)
page = doc[29]

variants = {
    "default 300dpi":      dict(dpi=300, config=""),
    "psm 6, 300dpi":       dict(dpi=300, config="--psm 6"),
    "psm 6, 400dpi":       dict(dpi=400, config="--psm 6"),
    "psm 4, 400dpi":       dict(dpi=400, config="--psm 4"),
}

for name, kw in variants.items():
    text = ocr_page_cfg(page, **kw)
    snippet = [ln for ln in text.splitlines() if "conspicuous" in ln or "not less than" in ln]
    print(f"--- {name} ---")
    for ln in snippet:
        print("   ", ln.strip())
    print()

doc.close()

--- default 300dpi ---
    (a) cause to be published in not less than two national daily newspapers
    (5) exhibit in a conspicuous position in each of its offices, branches and
    section is, in respect of each such failure, liable to a penalty of not less than
    commits an offence and is liable on conviction to a fine of not less than

--- psm 6, 300dpi ---
    (a) cause to be published in not less than two national daily newspapers
    (5) exhibit ina conspicuous position in each of its offices, branches and
    section is, in respect of each such failure, liable to a penalty of not less than
    commits an offence and is liable on conviction to a fine of not less than

--- psm 6, 400dpi ---
    (a) cause to be published in not less than two national daily newspapers
    (5) exhibit in a conspicuous position in each of its offices, branches and
    section is, in respect of each such failure, liable to a penalty of not less than
    commits an offence and is liable on conviction

In [8]:
import re
from collections import Counter

def find_repeated_lines(pages, min_fraction=0.4):
    """Lines that appear on many pages are headers or footers, not content."""
    counts = Counter()
    for p in pages:
        for line in {ln.strip() for ln in p["text"].splitlines() if ln.strip()}:
            if len(line) <= 90:
                counts[line] += 1
    threshold = max(3, int(len(pages) * min_fraction))
    return {line for line, c in counts.items() if c >= threshold}


NUMERIC_ONLY = re.compile(r"^[\s\d\-–—.,|]*$")
PAGE_MARKER = re.compile(r"^[A-Z]?\s?\d{1,4}$")

def clean_page(text, boilerplate):
    kept = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s in boilerplate:
            continue
        if NUMERIC_ONLY.match(s) or PAGE_MARKER.match(s):
            continue
        kept.append(s)

    out = "\n".join(kept)
    out = re.sub(r"(\w)-\n(\w)", r"\1\2", out)      # rejoin hyphenated words
    out = re.sub(r"\n{3,}", "\n\n", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out.strip()


# test on BOFIA page 30
data = json.loads((PROCESSED / (Path(bofia.name).stem + ".json")).read_text(encoding="utf-8"))
boilerplate = find_repeated_lines(data["pages"])

print("Boilerplate lines detected:", len(boilerplate))
for b in list(boilerplate)[:8]:
    print("   drop:", b[:70])

raw = data["pages"][29]["text"]
cleaned = clean_page(raw, boilerplate)

print(f"\nBefore: {len(raw)} chars | After: {len(cleaned)} chars\n")
print(cleaned[:900])


Boilerplate lines detected: 0

Before: 2525 chars | After: 2502 chars

Contents
and form of
accounts,
2020 No. § Banks and Other Financial Institutions Act, 2020
(2) Every bank or other financial institution shall thereafter, but not later
than seven days after approval for publication by the Bank—
(a) cause to be published in not less than two national daily newspapers
printed and circulating in Nigeria ;
(5) exhibit in a conspicuous position in each of its offices, branches and
website ; and
(c) forward to the Bank, copies of the bank's published Statement of
Financial Position and Statement of Profit and Loss and Other
Comprehensive Income duly signed with the full names of the directors
of the bank who signed the financial Statements, and in the case of a noninterest bank, a copy of the report of tts advisory committee of experts,
(3) Every published account of'a bank under subsection (2), shall disclose
in detail penalties paid for the contravention of


In [9]:
def normalise_for_match(line):
    """Strip digits and punctuation so OCR variants of the same header collide."""
    return " ".join(re.sub(r"[^a-z]+", " ", line.lower()).split())


def find_repeated_lines(pages, min_fraction=0.4):
    counts = Counter()
    for p in pages:
        seen = set()
        for ln in p["text"].splitlines():
            n = normalise_for_match(ln)
            if n and len(n) <= 90 and len(n.split()) >= 2:
                seen.add(n)
        counts.update(seen)
    threshold = max(3, int(len(pages) * min_fraction))
    return {n for n, c in counts.items() if c >= threshold}


# prefixes where the hyphen is part of the term, not a line break
KEEP_HYPHEN = {"non", "pre", "post", "anti", "self", "co", "re", "inter", "intra",
               "multi", "sub", "cross", "semi", "quasi", "counter", "micro", "e"}

def rejoin_hyphens(text):
    def repl(m):
        first, second = m.group(1), m.group(2)
        return f"{first}-{second}" if first.lower() in KEEP_HYPHEN else f"{first}{second}"
    return re.sub(r"([A-Za-z]{1,12})-\n([a-z]+)", repl, text)


def clean_page(text, boilerplate):
    kept = []
    for line in text.splitlines():
        s = line.strip()
        if not s:
            continue
        if normalise_for_match(s) in boilerplate:
            continue
        if NUMERIC_ONLY.match(s) or PAGE_MARKER.match(s):
            continue
        kept.append(s)

    out = rejoin_hyphens("\n".join(kept))
    out = re.sub(r"\n{3,}", "\n\n", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out.strip()


boilerplate = find_repeated_lines(data["pages"])
print("Boilerplate patterns detected:", len(boilerplate))
for b in list(boilerplate)[:10]:
    print("   drop:", b[:70])

cleaned = clean_page(data["pages"][29]["text"], boilerplate)
print(f"\nBefore: {len(raw)} chars | After: {len(cleaned)} chars\n")
print(cleaned[:900])

Boilerplate patterns detected: 0

Before: 2525 chars | After: 2503 chars

Contents
and form of
accounts,
2020 No. § Banks and Other Financial Institutions Act, 2020
(2) Every bank or other financial institution shall thereafter, but not later
than seven days after approval for publication by the Bank—
(a) cause to be published in not less than two national daily newspapers
printed and circulating in Nigeria ;
(5) exhibit in a conspicuous position in each of its offices, branches and
website ; and
(c) forward to the Bank, copies of the bank's published Statement of
Financial Position and Statement of Profit and Loss and Other
Comprehensive Income duly signed with the full names of the directors
of the bank who signed the financial Statements, and in the case of a non-interest bank, a copy of the report of tts advisory committee of experts,
(3) Every published account of'a bank under subsection (2), shall disclose
in detail penalties paid for the contravention o


In [10]:
counts = Counter()
for p in data["pages"]:
    seen = set()
    for ln in p["text"].splitlines():
        n = normalise_for_match(ln)
        if n:
            seen.add(n)
    counts.update(seen)

print("pages:", len(data["pages"]),
      "| 40% threshold:", max(3, int(len(data["pages"]) * 0.4)))
print("distinct normalised lines:", len(counts))
print()
for n, c in counts.most_common(15):
    print(f"{c:>4}  {n[:80]}")

pages: 88 | 40% threshold: 35
distinct normalised lines: 3276

  70  a
  33  banks and other financial institutions act no
  31  no banks and other financial institutions act
  16  act no
  11  and
   9  fund
   7  a no banks and other financial institutions act
   7  institutions
   7  the tribunal
   7  no
   7  of the
   6  lfn
   6  institution
   6  resolution
   6  tribunal


In [20]:
def line_key(line):
    """Order-independent fingerprint of a line: letters only, alphabetised."""
    return " ".join(sorted(re.sub(r"[^a-z]+", " ", line.lower()).split()))


def find_repeated_lines(pages, min_fraction=0.3):
    counts = Counter()
    for p in pages:
        seen = {line_key(ln) for ln in p["text"].splitlines()}
        seen.discard("")
        counts.update(seen)
    threshold = max(3, int(len(pages) * min_fraction))
    return {k for k, c in counts.items() if c >= threshold and len(k) <= 90}


def clean_page(text, boilerplate):
    lines = text.splitlines()
    idx = [i for i, l in enumerate(lines) if l.strip()]
    if not idx:
        return ""
    first, last = idx[0], idx[-1]

    kept = []
    for i, line in enumerate(lines):
        s = line.strip()
        if not s or line_key(s) in boilerplate:
            continue
        # A bare number at the very top or bottom of a page is a page number.
        # Mid-page it is a table cell: a capital requirement, a fee, a year.
        at_edge = i <= first + 1 or i >= last - 1
        if (NUMERIC_ONLY.match(s) or PAGE_MARKER.match(s)) and at_edge:
            continue
        kept.append(s)

    out = rejoin_hyphens("\n".join(kept))
    out = re.sub(r"\n{3,}", "\n\n", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out.strip()



boilerplate = find_repeated_lines(data["pages"])
print("Boilerplate patterns:", len(boilerplate))
for b in list(boilerplate)[:10]:
    print("   drop:", b[:75])

cleaned = clean_page(data["pages"][29]["text"], boilerplate)
print(f"\nBefore: {len(raw)} | After: {len(cleaned)}\n")
print(cleaned[:700])

Boilerplate patterns: 2
   drop: act and banks financial institutions no other
   drop: a

Before: 2525 | After: 2443

Contents
and form of
accounts,
(2) Every bank or other financial institution shall thereafter, but not later
than seven days after approval for publication by the Bank—
(a) cause to be published in not less than two national daily newspapers
printed and circulating in Nigeria ;
(5) exhibit in a conspicuous position in each of its offices, branches and
website ; and
(c) forward to the Bank, copies of the bank's published Statement of
Financial Position and Statement of Profit and Loss and Other
Comprehensive Income duly signed with the full names of the directors
of the bank who signed the financial Statements, and in the case of a non-interest bank, a copy of the report of tts advisory commi


In [21]:
cleaned_docs = {}
for path in sorted(PROCESSED.glob("*.json")):
    d = json.loads(path.read_text(encoding="utf-8"))
    bp = find_repeated_lines(d["pages"])
    pages = [{"page": p["page"], "method": p["method"], "text": clean_page(p["text"], bp)}
             for p in d["pages"]]
    d["pages"] = [p for p in pages if p["text"]]
    cleaned_docs[path.stem] = d

    before = sum(len(p["text"]) for p in json.loads(path.read_text(encoding="utf-8"))["pages"])
    after = sum(len(p["text"]) for p in d["pages"])
    pct = round(100 * (before - after) / before, 1) if before else 0
    flag = "  <-- CHECK" if pct > 25 else ""
    print(f"{pct:>5}% removed | {len(bp):>2} patterns | {d['title'][:45]}{flag}")

  2.9% removed |  2 patterns | Banks and Other Financial Institutions Act 20
  3.0% removed |  0 patterns | Guidelines on Liquidity Risk Management and I
  4.1% removed |  0 patterns | Guidelines on Regulatory Capital
  1.4% removed |  0 patterns | Re: Impact of Recent Policy Reforms-Prudentia
  5.2% removed |  0 patterns | Central Bank of Nigeria Risk-Based Cybersecur
  2.0% removed |  4 patterns | Issuance of Baseline Standards for Automated 
  9.0% removed |  6 patterns | Guidelines for Licensing of Banks and Other F
  2.0% removed |  2 patterns | Circular to All Banks and Other Financial Ins
  0.9% removed |  0 patterns | RE: GUIDELINES ON MANAGEMENT OF DORMANT ACCOU
  5.3% removed |  1 patterns | Guidance Note on Anti-Money Laundering and Co
  3.7% removed |  0 patterns | Corporate Governance Guidelines for Commercia
  1.0% removed |  0 patterns | Additional Know Your Customer Requirement in 
  3.8% removed |  0 patterns | Review of Minimum Capital Requirements for Co
  2.9% remov

In [22]:
sources.loc[sources["title"].str.startswith("ndic act"), "title"] = \
    "Nigeria Deposit Insurance Corporation Act 2023"
sources.loc[sources["title"].str.contains("SEC Consolidated JUNE2013"), "title"] = \
    "SEC Consolidated Rules and Regulations 2013"
sources.loc[sources["title"].str.contains("SEC AMLCFTCPF"), "title"] = \
    "SEC Capital Market Operators AML/CFT/CPF Regulations 2022"
sources.to_csv("../data/sources.csv", index=False)

for stem, d in cleaned_docs.items():
    match = sources.loc[sources["filename"] == d["filename"], "title"]
    if len(match):
        d["title"] = match.iloc[0]

print(sources["title"].to_string())

0     Review of Minimum Capital Requirements for Com...
1                      Guidelines on Regulatory Capital
2     Guidelines on Liquidity Risk Management and In...
3     Corporate Governance Guidelines for Commercial...
4     Central Bank of Nigeria Risk-Based Cybersecuri...
5     Issuance of Baseline Standards for Automated A...
6     Guidelines for Licensing of Banks and Other Fi...
7     Additional Know Your Customer Requirement in R...
8     RE: GUIDELINES ON MANAGEMENT OF DORMANT ACCOUN...
9     Re: Impact of Recent Policy Reforms-Prudential...
10       Nigeria Deposit Insurance Corporation Act 2023
11                               Supervisory Guidelines
12    SEC Capital Market Operators AML/CFT/CPF Regul...
13          SEC Consolidated Rules and Regulations 2013
14           New Rules and sundry amendments April 2025
15                              Executed Rules Dec 2024
16    Rules on Issuance Offering and Custody of Digi...
17    Rules Relating to the Complaints Managemen

In [23]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


def flatten_document(d):
    """Join pages into one string, remembering which page each offset came from."""
    parts, spans, cursor = [], [], 0
    for p in d["pages"]:
        parts.append(p["text"])
        spans.append((cursor, cursor + len(p["text"]), p["page"], p["method"]))
        cursor += len(p["text"]) + 2
    return "\n\n".join(parts), spans


def page_at(offset, spans):
    for start, end, page, method in spans:
        if start <= offset < end:
            return page, method
    return spans[-1][2], spans[-1][3]


def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split into overlapping pieces, preferring paragraph then sentence breaks."""
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            window = text[start:end]
            cut = max(window.rfind("\n\n"), window.rfind(". "), window.rfind("\n"))
            if cut > size * 0.5:
                end = start + cut + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append((start, piece))
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks


all_chunks = []
for stem, d in cleaned_docs.items():
    text, spans = flatten_document(d)
    for i, (offset, piece) in enumerate(split_text(text)):
        page, method = page_at(offset, spans)
        all_chunks.append({
            "chunk_id": f"{stem[:40]}__{i:04d}",
            "text": piece,
            "regulator": d["regulator"],
            "ref_no": d["ref_no"],
            "title": d["title"],
            "url": d["url"],
            "document_date": d["document_date"],
            "page": page,
            "extraction": method,
            "source_file": d["filename"],
        })

chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_json("../data/chunks.jsonl", orient="records", lines=True, force_ascii=False)

print("Chunks:", len(chunks_df))
print("Avg length:", int(chunks_df["text"].str.len().mean()))
print("\nBy regulator:")
print(chunks_df["regulator"].value_counts().to_string())
print("\nBy extraction method:")
print(chunks_df["extraction"].value_counts().to_string())
print("\n--- sample chunk ---")
s = chunks_df.iloc[len(chunks_df) // 2]
print(f"{s['title'][:60]} | page {s['page']} | {s['extraction']}\n")
print(s["text"][:600])

Chunks: 3828
Avg length: 761

By regulator:
regulator
SEC     2336
CBN     1173
NDIC     237
FGN       82

By extraction method:
extraction
text    3386
ocr      442

--- sample chunk ---
SEC Capital Market Operators AML/CFT/CPF Regulations 2022 | page 37 | text

company)
its Trustee (where it is a Unit Trust);
its managing (general) partner (where it is a limited partnership);
(e)
account signatories; and
(f)
any other person who has control over the relationship such as fund administrator or manager;
(7)
Where other investment vehicles are involved, the same steps under sub – regulation (6) above this
regulation shall be taken where it is appropriate to do so and all reasonable steps shall be taken to verify
the identity of the beneficial owners of the funds and of those who have control over the funds.
(8)
Intermediaries shall be treated as individu


In [24]:
def looks_like_index(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) < 5:
        return False
    short = sum(1 for l in lines if len(l) < 60)
    sentences = text.count(". ") + text.count(".\n")
    return (short / len(lines)) > 0.75 and sentences < len(lines) * 0.3


chunks_df["is_index"] = chunks_df["text"].apply(looks_like_index)

print("Flagged as index/contents:", int(chunks_df["is_index"].sum()),
      f"({round(100 * chunks_df['is_index'].mean(), 1)}%)")
print()
print(chunks_df.loc[chunks_df["is_index"]].groupby("title").size()
      .sort_values(ascending=False).head(8).to_string())

print("\n=== two flagged samples ===")
for _, r in chunks_df.loc[chunks_df["is_index"]].sample(2, random_state=0).iterrows():
    print(f"\n[{r['title'][:45]} p{r['page']}]")
    print(r["text"][:320])

print("\n=== two NOT flagged, for comparison ===")
for _, r in chunks_df.loc[~chunks_df["is_index"]].sample(2, random_state=0).iterrows():
    print(f"\n[{r['title'][:45]} p{r['page']}]")
    print(r["text"][:320])

Flagged as index/contents: 174 (4.5%)

title
SEC Consolidated Rules and Regulations 2013                                                                                                                                                                           34
Rules on Issuance Offering and Custody of Digital Assets                                                                                                                                                              20
Central Bank of Nigeria Risk-Based Cybersecurity Framework and Guidelines for Deposit Money Banks and Payment Service Banks                                                                                           15
Circular to All Banks and Other Financial Institutions: Uniform Account Opening Forms and Minimum Information Requirements for Three-Tiered KYC for Customers of Banks and Other Financial Institutions in Nigeria    15
Issuance of Baseline Standards for Automated Anti-Money Laundering (AML) Solution for F

In [25]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBED_MODEL)

v = embedder.encode("minimum capital requirement", normalize_embeddings=True)
print("Dimensions:", len(v))

probe = embedder.encode([
    "The minimum paid-up capital for a commercial bank with international authorisation is 500 billion naira.",
    "Banks shall maintain adequate regulatory capital at all times.",
    "The cat sat on the mat in the afternoon sun.",
], normalize_embeddings=True)

print("capital vs capital:", round(float(probe[0] @ probe[1]), 3))
print("capital vs cat    :", round(float(probe[0] @ probe[2]), 3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dimensions: 384
capital vs capital: 0.613
capital vs cat    : 0.276


In [26]:
import chromadb

texts = chunks_df["text"].tolist()
embeddings = embedder.encode(
    texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
).tolist()

meta_cols = ["chunk_id", "regulator", "ref_no", "title", "url",
             "document_date", "page", "extraction", "source_file"]
metadatas = chunks_df[meta_cols].fillna("").to_dict("records")
for m in metadatas:
    m["page"] = int(m["page"])

ids = [f"c{i:06d}" for i in range(len(chunks_df))]

client = chromadb.PersistentClient(path="../chroma_db")
if "nigeria_reg" in [c.name for c in client.list_collections()]:
    client.delete_collection("nigeria_reg")
collection = client.create_collection("nigeria_reg", metadata={"hnsw:space": "cosine"})

B = 500
for i in range(0, len(ids), B):
    collection.add(
        ids=ids[i:i+B],
        documents=texts[i:i+B],
        embeddings=embeddings[i:i+B],
        metadatas=metadatas[i:i+B],
    )

print("Indexed:", collection.count())

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Indexed: 3828


In [28]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def search_dense(question, k=5):
    """Semantic search: find chunks whose meaning is closest to the question."""
    qv = embedder.encode(QUERY_PREFIX + question, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[qv], n_results=k)

    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({**meta, "text": doc, "score": round(1 - dist, 3)})
    return hits


def show(question, hits):
    print(f"\nQ: {question}")
    for i, h in enumerate(hits, 1):
        flag = " [OCR]" if h["extraction"] == "ocr" else ""
        print(f"  {i}. {h['score']}  {h['regulator']} | {h['title'][:52]} p{h['page']}{flag}")
        print(f"      {h['text'][:150].replace(chr(10), ' ')}...")


for q in [
    "What is the minimum paid-up share capital for a commercial bank with international authorisation?",
    "What are the board composition requirements for banks?",
    "How long must a bank keep customer identification records?",
]:
    show(q, search_dense(q))


Q: What is the minimum paid-up share capital for a commercial bank with international authorisation?
  1. 0.804  CBN | Review of Minimum Capital Requirements for Commercia p4
      h of the economy. 4. Which category of banks are affected by the Programme? The Programme shall apply to commercial, merchant, and non-interest banks....
  2. 0.761  CBN | Banks and Other Financial Institutions Act 2020 p3 [OCR]
      e capital of banks and compliance with the minimum paid-up share capital requirement. Shareholder’s voting rights to be proportional to shareholding. ...
  3. 0.754  SEC | Rules on Issuance Offering and Custody of Digital As p9
      l and fidelity bond Evidence of Required Minimum Paid up Capital – N500,000,000 (Five hundred Million Naira only) (i.e. Bank balances, Fixed asset or ...
  4. 0.742  CBN | Banks and Other Financial Institutions Act 2020 p26 [OCR]
      20.—(1) Subject to the approval of the Bank, a bank may acquire or hold part of the share capital of any agricult

In [21]:
import requests
from urllib.parse import quote

BROWSER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
BASE = "https://www.cbn.gov.ng"


def fetch_json(url, timeout=60):
    r = requests.get(url, headers={"User-Agent": BROWSER_UA}, timeout=timeout)
    r.raise_for_status()
    return r.json()


def normalise_cbn(records):
    out = []
    for r in records:
        link = (r.get("link") or "").strip()
        if not link.lower().endswith(".pdf"):
            continue
        out.append({
            "regulator": "CBN",
            "ref_no": (r.get("refNo") or "").strip(),
            "title": " ".join((r.get("title") or "").split()),
            "keywords": " ".join((r.get("keywords") or "").split()),
            "url": BASE + quote(link.upper()),
            "date": (r.get("documentDate") or "").strip(),
        })
    return out


records = (normalise_cbn(fetch_json(f"{BASE}/api/GetSupervisionCirculars?format=json"))
           + normalise_cbn(fetch_json(f"{BASE}/api/GetAllCirculars?format=json")))

df = pd.DataFrame(records).drop_duplicates(subset="url").reset_index(drop=True)
df["date_parsed"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
df = df.sort_values("date_parsed", ascending=False).reset_index(drop=True)

print("CBN records across both endpoints:", len(df))

gap_terms = [
    "anti-money laundering", "money laundering", "customer due diligence",
    "know your customer", "record keeping", "retention",
    "politically exposed", "suspicious transaction", "terrorism financing",
    "aml/cft", "amlcft",
]
pattern = "|".join(gap_terms)

haystack = df["title"] + " || " + df["keywords"].fillna("")
candidates = df[haystack.str.contains(pattern, case=False, na=False, regex=True)]

print("Candidates:", len(candidates), "\n")
for i, r in candidates.head(25).iterrows():
    date = r["date_parsed"].date() if pd.notna(r["date_parsed"]) else "no date"
    print(f"[{i}] {date} | {r['ref_no']}")
    print(f"     {r['title'][:95]}")

CBN records across both endpoints: 2590
Candidates: 43 

[13] 2026-03-31 | CMD/DIR/PUB/CIR/001006
     Implementation of the Baseline Standards for Automated AML/CFT/CPF Solutions
[25] 2026-03-10 | BSD/DIR/PUB/LAB/019/002
     Issuance of Baseline Standards for Automated Anti-Money Laundering (AML) Solution for Financial
[44] 2025-05-21 | BSD/DIR/CON/AML/018/033
     Exposure of Draft Baseline Standards for Automated Anti-Money Laundering (AML) Solutions
[47] 2025-04-17 | CMD/DIR/INT/GEN/001/001
     RE: Compliance with AML/CFT/CPF Regulations
[87] 2024-06-30 | FPR/DIR/PUB/CIR/001/003
     Money Laundering, Terrorism Financing and Proliferation Financing: Risk Assessment Report of Ba
[128] 2023-12-08 | FPR/DIR/PUB/CIR/002/002
     Additional Know Your Customer Requirement in Respect of Non-Profit Organizations
[143] 2023-06-23 | FPR/DIR/PUB/CIR/007/075
     Guidance Note on Politically Exposed Persons (PEP)
[156] 2023-02-07 | FPR/DIR/PUB/CIR/001/069
     Circular to All Banks and Other

In [29]:
from urllib.parse import urlparse

# drop the failed entries from the manifest
sources = sources[~sources["status"].astype(str).str.startswith("FAILED")].reset_index(drop=True)
print("Cleaned corpus:", len(sources), "documents")

Cleaned corpus: 24 documents


In [25]:
import hashlib, time

NEW_ROWS = [181, 143, 411, 881]   # edit this list

def make_filename(regulator, ref_no, title, url, max_len=70):
    stem = f"{regulator}_{ref_no or 'NA'}_{title}"
    stem = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_")[:max_len]
    return f"{stem}_{hashlib.sha1(url.encode()).hexdigest()[:8]}.pdf"


def download_pdf(url, destination, timeout=180):
    if destination.exists():
        return False
    parsed = urlparse(url)
    r = requests.get(url, timeout=timeout, headers={
        "User-Agent": BROWSER_UA, "Referer": f"{parsed.scheme}://{parsed.netloc}/"})
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError(f"Not a PDF: {r.headers.get('Content-Type')}")
    destination.write_bytes(r.content)
    return True


added = []
for idx in NEW_ROWS:
    row = df.loc[idx]
    fname = make_filename(row["regulator"], row["ref_no"], row["title"], row["url"])
    dest = RAW_DIR / fname
    try:
        download_pdf(row["url"], dest)
        size_kb = round(dest.stat().st_size / 1024, 1)
        status = "CHECK - tiny" if size_kb < 30 else "ok"
    except Exception as exc:
        size_kb, status = 0, f"FAILED {type(exc).__name__}: {exc}"[:90]

    added.append({"filename": fname, "regulator": row["regulator"], "ref_no": row["ref_no"],
                  "title": row["title"], "url": row["url"], "document_date": row["date"],
                  "size_kb": size_kb, "status": status, "error": "",
                  "downloaded_on": pd.Timestamp.today().date().isoformat()})
    print(f"{status:<14}{size_kb:>8} KB  {row['title'][:55]}")
    time.sleep(2)

sources = (pd.concat([sources, pd.DataFrame(added)], ignore_index=True)
             .drop_duplicates(subset="url", keep="last").reset_index(drop=True))
sources.to_csv("../data/sources.csv", index=False)
print("\nCorpus now:", len(sources), "documents")


ok               595.7 KB  Guidance Note on Anti-Money Laundering and Combating th
ok               338.1 KB  Guidance Note on Politically Exposed Persons (PEP)
ok              1114.3 KB  Anti-Money Laundering/Combating the Financing of Terror
ok              1535.0 KB  Circular to All Banks and Other Financial Institutions:

Corpus now: 24 documents


In [19]:
import json, collections
rows = [json.loads(l) for l in open("../data/chunks.jsonl", encoding="utf-8")]
docs = collections.Counter(r["source_file"] for r in rows)
print("chunks:", len(rows), "| docs:", len(docs))
print("in index:", collection.count())


chunks: 3810 | docs: 24
in index: 3810
